# Orion Dense vs Sparse Experiment Notebook (Colab Ready)

This notebook is designed for controlled, reproducible comparison of dense and sparse attention in Orion.
It focuses on:

- Throughput and step time
- VRAM footprint
- Optimization behavior (loss / ppl / grad norm)
- Sparse structural diagnostics (`valid_neighbors`, `future_edges`, `duplicate_edges`, etc.)

It is intentionally built as a research notebook, not an experimentation framework package.


## Experimental Philosophy

To identify where sparse beats dense, keep comparisons fair:

- Same model architecture, optimizer, and data source.
- Sweep only the variables you care about (`seq_len`, sparse `window_size`, `expander_degree`, seed).
- Compare by matched `(seq_len, seed)`.
- Use sparse diagnostics to ensure you are not accidentally running a near-dense sparse config.

Defaults below are tuned for a practical Colab GPU run and can be scaled up.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/akashkguw/orion.git"
REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()

if IN_COLAB and not REPO_ROOT.exists():
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")


In [ ]:
import sys
import subprocess

# Core project deps
subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    "requirements.txt",
    "-r",
    "requirements-dev.txt",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# Analysis deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "seaborn"], check=True)

print("Environment ready.")


In [ ]:
import json
import math
import time
import itertools
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)


## Configure Sweep

Use `PROFILE='pilot'` first. Switch to `PROFILE='full'` for heavier sweeps.

Notes:
- Sparse configs with `window_size >= seq_len` are excluded automatically.
- Dense runs use the same model/data/optimizer settings for fairness.


In [ ]:
# ----------------------------
# Experiment control panel
# ----------------------------
PROFILE = "pilot"  # "pilot" or "full"

if PROFILE == "pilot":
    STEPS = 200
    SEEDS = [123]
    SEQ_LENS = [256, 512, 1024]
    BATCH_BY_SEQ = {256: 16, 512: 8, 1024: 4}
    SPARSE_GRID = [
        (64, 8),
        (128, 16),
    ]
elif PROFILE == "full":
    STEPS = 400
    SEEDS = [123, 456]
    SEQ_LENS = [256, 512, 1024, 2048]
    BATCH_BY_SEQ = {256: 16, 512: 8, 1024: 4, 2048: 2}
    SPARSE_GRID = [
        (64, 8),
        (128, 16),
        (256, 16),
    ]
else:
    raise ValueError(f"Unknown PROFILE: {PROFILE}")

BASE_MODEL = {
    "name": "orion",
    "d_model": 256,
    "n_layers": 4,
    "n_heads": 4,
    "mlp_mult": 4,
}

OPTIM = {"lr": 3e-4}

EXPERIMENT_ID = time.strftime("dense_sparse_grid_%Y%m%d_%H%M%S")
RUNS_ROOT = Path("runs") / EXPERIMENT_ID
CFG_ROOT = Path("configs") / "generated" / EXPERIMENT_ID
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
CFG_ROOT.mkdir(parents=True, exist_ok=True)

print("PROFILE:", PROFILE)
print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("RUNS_ROOT:", RUNS_ROOT)


In [ ]:
@dataclass(frozen=True)
class TrialSpec:
    backend: str
    seq_len: int
    batch_size: int
    seed: int
    steps: int
    window_size: int | None = None
    expander_degree: int | None = None
    d_model: int = BASE_MODEL["d_model"]
    n_layers: int = BASE_MODEL["n_layers"]
    n_heads: int = BASE_MODEL["n_heads"]
    mlp_mult: int = BASE_MODEL["mlp_mult"]
    lr: float = OPTIM["lr"]

    @property
    def sparse_tag(self) -> str:
        if self.backend != "sparse":
            return "dense"
        return f"w{self.window_size}_d{self.expander_degree}"

    @property
    def trial_id(self) -> str:
        base = f"{self.backend}_T{self.seq_len}_bs{self.batch_size}_seed{self.seed}"
        if self.backend == "sparse":
            base += f"_w{self.window_size}_d{self.expander_degree}"
        return base


def build_trial_specs() -> list[TrialSpec]:
    specs: list[TrialSpec] = []

    for seed, seq_len in itertools.product(SEEDS, SEQ_LENS):
        batch_size = BATCH_BY_SEQ[seq_len]

        # Dense baseline
        specs.append(
            TrialSpec(
                backend="dense",
                seq_len=seq_len,
                batch_size=batch_size,
                seed=seed,
                steps=STEPS,
            )
        )

        # Sparse variants (only if window < seq_len)
        for window_size, expander_degree in SPARSE_GRID:
            if window_size >= seq_len:
                continue
            specs.append(
                TrialSpec(
                    backend="sparse",
                    seq_len=seq_len,
                    batch_size=batch_size,
                    seed=seed,
                    steps=STEPS,
                    window_size=window_size,
                    expander_degree=expander_degree,
                )
            )

    return specs


TRIAL_SPECS = build_trial_specs()
trial_df = pd.DataFrame(asdict(s) for s in TRIAL_SPECS)
print(f"Planned runs: {len(TRIAL_SPECS)}")
trial_df.head(20)


## Runner Utilities

- Writes one config per run.
- Executes `python -m orion.train`.
- Parses `metrics.jsonl` into a compact summary row.


In [ ]:
def trial_to_config(spec: TrialSpec, out_dir: Path) -> dict:
    cfg = {
        "run": {
            "out_dir": str(out_dir),
            "seed": int(spec.seed),
            "steps": int(spec.steps),
            "log_every": 10,
            "save_every": int(spec.steps),
        },
        "data": {
            "dataset": "tinyshakespeare",
            "root": "data",
            "seq_len": int(spec.seq_len),
            "batch_size": int(spec.batch_size),
        },
        "model": {
            "name": "orion",
            "d_model": int(spec.d_model),
            "n_layers": int(spec.n_layers),
            "n_heads": int(spec.n_heads),
            "mlp_mult": int(spec.mlp_mult),
        },
        "attention": {
            "backend": spec.backend,
        },
        "optim": {
            "lr": float(spec.lr),
        },
    }
    if spec.backend == "sparse":
        cfg["attention"]["window_size"] = int(spec.window_size)
        cfg["attention"]["expander_degree"] = int(spec.expander_degree)
    return cfg


def load_metrics_rows(metrics_path: Path) -> list[dict]:
    rows: list[dict] = []
    if not metrics_path.exists():
        return rows
    with metrics_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def summarize_trial(spec: TrialSpec, run_dir: Path, duration_s: float, status: str, error: str = "") -> dict:
    metrics_path = run_dir / "metrics.jsonl"
    rows = load_metrics_rows(metrics_path)

    out = {
        "trial_id": spec.trial_id,
        "backend": spec.backend,
        "sparse_tag": spec.sparse_tag,
        "seq_len": spec.seq_len,
        "batch_size": spec.batch_size,
        "seed": spec.seed,
        "steps_target": spec.steps,
        "window_size": spec.window_size,
        "expander_degree": spec.expander_degree,
        "duration_s": duration_s,
        "status": status,
        "error": error,
        "run_dir": str(run_dir),
    }

    if status != "ok" or not rows:
        return out

    step_rows = [r for r in rows if r.get("type") == "step"]
    window_rows = [r for r in rows if r.get("type") == "window"]

    if not step_rows:
        out["status"] = "no_step_rows"
        return out

    final_step = step_rows[-1]
    tail = step_rows[-min(20, len(step_rows)):]

    def mean_key(xs, key):
        vals = [float(x[key]) for x in xs if key in x]
        return float(np.mean(vals)) if vals else math.nan

    out.update(
        {
            "steps_observed": int(final_step.get("step", len(step_rows))),
            "final_loss": float(final_step.get("loss", math.nan)),
            "final_ppl": float(final_step.get("ppl", math.nan)),
            "final_acc_top1": float(final_step.get("accuracy_top1", math.nan)),
            "throughput_tok_s": float(final_step.get("throughput_tokens_per_sec", math.nan)),
            "throughput_tok_s_tail_mean": mean_key(tail, "throughput_tokens_per_sec"),
            "step_time_ms_tail_mean": mean_key(tail, "step_time_ms"),
            "grad_norm_tail_mean": mean_key(tail, "grad_norm"),
        }
    )

    if window_rows:
        w = window_rows[-1]
        out.update(
            {
                "vram_peak_mib": float(w.get("vram_peak_mib", math.nan)),
                "attn_entropy": float(w.get("attention_entropy", math.nan)),
                "attn_entropy_norm": float(w.get("attention_entropy_normalized", math.nan)),
                "valid_neighbors": float(w.get("valid_neighbor_fraction", math.nan)),
                "window_mass_pct": float(w.get("attention_mass_window_pct", math.nan)),
                "expander_mass_pct": float(w.get("attention_mass_expander_pct", math.nan)),
                "valid_neighbor_fraction_causal_cap": float(
                    w.get("valid_neighbor_fraction_causal_cap", math.nan)
                ),
                "valid_neighbor_fraction_vs_causal_cap": float(
                    w.get("valid_neighbor_fraction_vs_causal_cap", math.nan)
                ),
                "future_neighbor_slots": float(w.get("future_neighbor_slots", math.nan)),
                "duplicate_neighbor_slots": float(w.get("duplicate_neighbor_slots", math.nan)),
            }
        )

    return out


def run_trial(spec: TrialSpec, overwrite: bool = False) -> dict:
    run_dir = RUNS_ROOT / spec.trial_id
    cfg_path = CFG_ROOT / f"{spec.trial_id}.yaml"
    run_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = run_dir / "metrics.jsonl"
    if metrics_path.exists() and not overwrite:
        print(f"[skip] {spec.trial_id} (metrics exists)")
        return summarize_trial(spec, run_dir, duration_s=0.0, status="ok")

    cfg = trial_to_config(spec, run_dir)
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

    cmd = [sys.executable, "-m", "orion.train", "--config", str(cfg_path)]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=Path.cwd(), text=True, capture_output=True)
    dt = time.time() - t0

    (run_dir / "stdout.log").write_text(proc.stdout, encoding="utf-8")
    (run_dir / "stderr.log").write_text(proc.stderr, encoding="utf-8")

    if proc.returncode != 0:
        err_tail = "\n".join(proc.stderr.splitlines()[-30:])
        print(f"[fail] {spec.trial_id}\n{err_tail}")
        return summarize_trial(spec, run_dir, duration_s=dt, status="failed", error=err_tail)

    print(f"[ok] {spec.trial_id} ({dt:.1f}s)")
    return summarize_trial(spec, run_dir, duration_s=dt, status="ok")


In [ ]:
# Run all trials (set OVERWRITE=True to rerun existing outputs)
OVERWRITE = False

results = []
total = len(TRIAL_SPECS)

for i, spec in enumerate(TRIAL_SPECS, start=1):
    print(f"\n[{i}/{total}] {spec.trial_id}")
    row = run_trial(spec, overwrite=OVERWRITE)
    results.append(row)

summary_df = pd.DataFrame(results)
summary_path = RUNS_ROOT / "summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"\nSaved summary: {summary_path}")
print(summary_df["status"].value_counts(dropna=False))
summary_df.head(10)


## Aggregation and Comparative Views


In [ ]:
df = pd.read_csv(RUNS_ROOT / "summary.csv")
df_ok = df[df["status"] == "ok"].copy()

# Variant label for plotting
df_ok["variant"] = np.where(
    df_ok["backend"] == "dense",
    "dense",
    "sparse_" + df_ok["sparse_tag"],
)

agg = (
    df_ok.groupby(["backend", "variant", "seq_len"], as_index=False)
    .agg(
        runs=("trial_id", "count"),
        final_ppl_mean=("final_ppl", "mean"),
        final_ppl_std=("final_ppl", "std"),
        tput_mean=("throughput_tok_s_tail_mean", "mean"),
        tput_std=("throughput_tok_s_tail_mean", "std"),
        vram_mean=("vram_peak_mib", "mean"),
        vram_std=("vram_peak_mib", "std"),
        valid_vs_cap_mean=("valid_neighbor_fraction_vs_causal_cap", "mean"),
        expander_mass_mean=("expander_mass_pct", "mean"),
    )
    .sort_values(["seq_len", "backend", "variant"])
)

agg


In [ ]:
# Throughput vs sequence length
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_ok,
    x="seq_len",
    y="throughput_tok_s_tail_mean",
    hue="variant",
    marker="o",
    estimator="mean",
    errorbar="sd",
)
plt.title("Throughput (tail mean) vs Sequence Length")
plt.ylabel("tokens / sec")
plt.xlabel("seq_len")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# VRAM vs sequence length
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_ok,
    x="seq_len",
    y="vram_peak_mib",
    hue="variant",
    marker="o",
    estimator="mean",
    errorbar="sd",
)
plt.title("Peak VRAM vs Sequence Length")
plt.ylabel("MiB")
plt.xlabel("seq_len")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Quality (ppl) vs sequence length
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=df_ok,
    x="seq_len",
    y="final_ppl",
    hue="variant",
    marker="o",
    estimator="mean",
    errorbar="sd",
)
plt.title("Final PPL vs Sequence Length")
plt.ylabel("final ppl")
plt.xlabel("seq_len")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Matched dense vs best-sparse comparison per (seq_len, seed)
dense = df_ok[df_ok["backend"] == "dense"][
    ["seq_len", "seed", "throughput_tok_s_tail_mean", "final_ppl", "vram_peak_mib"]
].rename(
    columns={
        "throughput_tok_s_tail_mean": "dense_tput",
        "final_ppl": "dense_ppl",
        "vram_peak_mib": "dense_vram",
    }
)

sparse = df_ok[df_ok["backend"] == "sparse"].copy()
sparse_best = (
    sparse.sort_values(["seq_len", "seed", "throughput_tok_s_tail_mean"], ascending=[True, True, False])
    .groupby(["seq_len", "seed"], as_index=False)
    .first()
)[["seq_len", "seed", "variant", "throughput_tok_s_tail_mean", "final_ppl", "vram_peak_mib"]].rename(
    columns={
        "variant": "best_sparse_variant",
        "throughput_tok_s_tail_mean": "sparse_tput",
        "final_ppl": "sparse_ppl",
        "vram_peak_mib": "sparse_vram",
    }
)

cmp = dense.merge(sparse_best, on=["seq_len", "seed"], how="inner")
cmp["sparse_vs_dense_tput"] = cmp["sparse_tput"] / cmp["dense_tput"]
cmp["sparse_vs_dense_vram"] = cmp["sparse_vram"] / cmp["dense_vram"]
cmp["ppl_delta_sparse_minus_dense"] = cmp["sparse_ppl"] - cmp["dense_ppl"]

cmp


## How To Interpret

1. If `valid_neighbor_fraction_vs_causal_cap` is near 1.0 and `future/duplicate` are near 0, your sparse graph is structurally healthy.
2. At short contexts (e.g., 256), dense may still win due to highly optimized kernels.
3. Crossover search: increase `seq_len` while keeping sparse degree (`window + expander`) bounded.
4. To compare quality fairly, use same seed and same training budget (`steps`, model, optimizer).

Artifacts from this notebook are saved under:

- `runs/<EXPERIMENT_ID>/...`
- `configs/generated/<EXPERIMENT_ID>/...`
